<a href="https://colab.research.google.com/github/dmontillao/ciencia-datos-curso/blob/main/tarea_03_data_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer

In [2]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Parte 1 — Exploración y selección

In [3]:
print(df.columns)
print(df.shape)
print(df.size)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')
(891, 12)
10692


P2. Usa .loc para seleccionar las filas 10 a 15 (inclusive) y solo las columnas Name, Age y Survived.

In [4]:
df.loc[:15][['Name', 'Age', 'Survived']]

,Name,Age,Survived
0,"Braund, Mr. Owen Harris",22.0,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1
2,"Heikkinen, Miss. Laina",26.0,1
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1
4,"Allen, Mr. William Henry",35.0,0
5,"Moran, Mr. James",NaN,0
6,"McCarthy, Mr. Timothy J",54.0,0
7,"Palsson, Master. Gosta Leonard",2.0,0
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",27.0,1
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.0,1


Parte 2 — Valores nulos y limpieza
P3. Cuenta cuántos valores nulos hay en cada columna. ¿Cuáles columnas tienen nulos?

In [5]:
missing = (
    df.isna()
      .sum()
      .to_frame("n_missing")
      .assign(pct_missing=lambda x: 100 * x["n_missing"] / len(df))
      .sort_values("pct_missing", ascending=False)
)

duplicates = df.duplicated().sum()

display(missing)
print(f"Registros duplicados: {duplicates}")

,n_missing,pct_missing
Cabin,687,77.104377
Age,177,19.865320
Embarked,2,0.224467
PassengerId,0,0.000000
Name,0,0.000000
Pclass,0,0.000000
Survived,0,0.000000
Sex,0,0.000000
Parch,0,0.000000
SibSp,0,0.000000


Registros duplicados: 0


P4. Rellena los valores nulos de la columna Age con la mediana de esa columna.
Verifica que ya no haya nulos en Age.

In [8]:
df_eda = df.copy()
num_imputer = SimpleImputer(strategy="median")
df_eda["Age"] = num_imputer.fit_transform(df_eda[["Age"]]).ravel()

print("Después del tratamiento:")
print(f"Filas: {df_eda.shape[0]}")
print(f"Duplicados: {df_eda.duplicated().sum()}")
print("\nFaltantes restantes:")
display(df_eda.isna().sum().to_frame("n_missing"))

Después del tratamiento:
Filas: 891
Duplicados: 0

Faltantes restantes:


,n_missing
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


Parte 3 — Nuevas columnas y agrupaciones
P5. Crea una nueva columna llamada FamilySize que sea la suma de SibSp + Parch + 1 (el pasajero mismo).

In [9]:
df_eda["FamilySize"] = df_eda["SibSp"] + df_eda["Parch"] + 1
df_eda.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,2
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1


P6. Usando groupby, calcula la tasa de supervivencia promedio (columna Survived) por clase (Pclass).
¿Qué clase tuvo mayor probabilidad de sobrevivir?

In [11]:
df_eda.groupby("Pclass")["Survived"].mean()


,Survived
Pclass,
1,0.629630
2,0.472826
3,0.242363



P6. Usando groupby, calcula la tasa de supervivencia promedio (columna Survived) por clase (Pclass).
¿Qué clase tuvo mayor probabilidad de sobrevivir?

In [13]:
df_eda.groupby(["Sex", "Pclass"])[['Age', 'Fare']].agg('mean')

Age        Fare
Sex    Pclass                       
female 1       33.978723  106.125798
       2       28.703947   21.970121
       3       23.572917   16.118810
male   1       38.995246   67.226127
       2       30.512315   19.741782
       3       26.911873   12.661633

la clase con mayor supervivencia fue: clase 1

Parte 4 — Ordenamiento y duplicados
. Ordena el DataFrame por Fare de mayor a menor y muestra las primeras 5 filas.
¿Cuánto pagó el pasajero más caro?

In [14]:
df_eda['Fare'].sort_values(ascending=False).head()

,Fare
679,512.3292
258,512.3292
737,512.3292
88,263.0000
438,263.0000


P9. Verifica si hay filas duplicadas en el DataFrame. ¿Cuántas hay?

In [15]:
df_eda.duplicated().sum()

np.int64(0)